# Week 55, Day 4: Counting and combinatorics

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w055d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from math import comb, perm, factorial

print(factorial(5), perm(5, 2), comb(5, 2))
print([comb(4, k) for k in range(5)], sum(comb(10, k) for k in range(11)))
print(comb(52, 5), len(str(factorial(1000))))

In [ ]:
P = 10 ** 9 + 7

class Binomials:
    def __init__(self, limit, p=P):
        self.p = p
        self.fact = [1] * (limit + 1)
        for i in range(1, limit + 1):
            self.fact[i] = self.fact[i - 1] * i % p
        self.inv_fact = [1] * (limit + 1)
        self.inv_fact[limit] = pow(self.fact[limit], -1, p)
        for i in range(limit, 0, -1):
            self.inv_fact[i - 1] = self.inv_fact[i] * i % p

    def comb(self, n, k):
        if not 0 <= k <= n:
            return 0
        return self.fact[n] * self.inv_fact[k] % self.p * self.inv_fact[n - k] % self.p

b = Binomials(10 ** 6)
from math import comb
print(b.comb(10, 3), b.comb(10 ** 6, 500_000), comb(10 ** 6, 500_000) % P)

In [ ]:
from math import comb

def share(sweets, children, at_least=0):
    sweets -= at_least * children
    if sweets < 0:
        return 0
    return comb(sweets + children - 1, children - 1)

print(share(5, 3), share(5, 3, at_least=1), share(10, 4, at_least=2))

In [ ]:
from math import lcm

def divisible_by_none(N, divisors):
    total = 0
    m = len(divisors)
    for mask in range(1 << m):
        chosen = [divisors[i] for i in range(m) if mask >> i & 1]
        sign = -1 if len(chosen) % 2 else 1
        total += sign * (N // lcm(*chosen)) if chosen else N
    return total

print(divisible_by_none(100, [2, 3, 5]), sum(1 for x in range(1, 101) if x % 2 and x % 3 and x % 5))

In [ ]:
from math import factorial, comb

def rank(perm):
    items = sorted(perm)
    r = 0
    for i, x in enumerate(perm):
        smaller = items.index(x)
        r += smaller * factorial(len(perm) - 1 - i)
        items.pop(smaller)
    return r

print([comb(2 * n, n) // (n + 1) for n in range(8)])
print(rank("abc"), rank("acb"), rank("cba"), rank([3, 1, 4, 2]))

> **Key idea.** Ask whether order matters and whether things repeat: P(n, k), C(n, k), nᵏ, or C(n + k − 1, k) for multisets. Pascal's rule C(n, k) = C(n − 1, k − 1) + C(n − 1, k) builds rows with additions; c = c × (n − i) // (i + 1) computes one exactly; mod a prime, factorial and inverse-factorial tables give each binomial in O(1). Stars and bars share n identical things among k: C(n + k − 1, k − 1). Inclusion-exclusion counts what avoids every property by alternating signs over subsets of properties, as for derangements. Catalan numbers C(2n, n) / (n + 1) count balanced brackets and tree shapes, and the factorial number system ranks and unranks permutations.

### Exercise: Pascal by hand

Write `pascal_row(n)`: row n of Pascal's triangle as a list, built from row 0 with Pascal's rule (no `comb` or factorials). Then `binomial(n, k)`: C(n, k) exactly with the multiply-then-divide loop, using the smaller of k and n − k, and 0 when k is outside 0 to n.

In [ ]:
def pascal_row(n):
    ___

def binomial(n, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from math import comb\ndef test_pascal():\n    \"Pascal\"\n    assert pascal_row(0) == [1] and pascal_row(4) == [1, 4, 6, 4, 1]\n    assert all(pascal_row(n) == [comb(n, k) for k in range(n + 1)] for n in range(40))\n    assert binomial(5, 2) == 10 and binomial(5, 7) == 0 and binomial(5, -1) == 0 and binomial(0, 0) == 1\n    for n in range(0, 300, 7):\n        for k in range(0, n + 1, 5):\n            assert binomial(n, k) == comb(n, k)\n    assert binomial(10 ** 6, 3) == comb(10 ** 6, 3)\n    assert \"comb(\" not in source.split(\"def pascal_row\")[1] and \"factorial\" not in source")

### Exercise: Counting mod a prime

Write a class `Counter` with `Counter(limit, p=10**9 + 7)` precomputing factorials and inverse factorials mod p up to `limit` (one `pow` for the inverse of the largest, then downwards), and methods `comb(n, k)`, `perm(n, k)` and `multiset(n, k)` (choosing k from n kinds with repetition: C(n + k − 1, k)), all mod p, returning 0 outside the valid range. Raise `ValueError` if a needed factorial is beyond `limit`.

In [ ]:
class Counter:
    def __init__(self, limit, p=10 ** 9 + 7):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from math import comb, perm\nimport random\ndef test_modular():\n    \"factorial tables\"\n    c = Counter(1000)\n    assert c.comb(10, 3) == 120 and c.perm(5, 2) == 20 and c.multiset(3, 2) == 6 and c.comb(4, 5) == 0\n    rng = random.Random(1)\n    for _ in range(300):\n        n, k = rng.randint(0, 600), rng.randint(0, 600)\n        assert c.comb(n, k) == comb(n, k) % (10 ** 9 + 7) and c.perm(n, k) == (perm(n, k) if k <= n else 0) % (10 ** 9 + 7)\n        if n >= 1 and n + k - 1 <= 1000:\n            assert c.multiset(n, k) == comb(n + k - 1, k) % (10 ** 9 + 7)\n    small = Counter(12, 13)\n    assert small.comb(10, 4) == comb(10, 4) % 13\n    try:\n        c.comb(2000, 3)\n        assert False\n    except ValueError:\n        pass\n    big = Counter(10 ** 6)\n    assert big.comb(10 ** 6, 123456) == comb(10 ** 6, 123456) % (10 ** 9 + 7)")

### Exercise: Sharing out

Write `share(items, people, at_least=0)`: the ways to share identical items among people with each getting at least `at_least` (stars and bars), and `share_capped(items, people, cap)`: the same, but nobody may get **more** than `cap`, by inclusion-exclusion over how many people break the cap: Σ (−1)ʲ C(people, j) × share(items − j × (cap + 1), people).

In [ ]:
from math import comb

def share(items, people, at_least=0):
    ___

def share_capped(items, people, cap):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools\ndef brute(items, people, lo, hi):\n    return sum(1 for xs in itertools.product(range(lo, hi + 1), repeat=people) if sum(xs) == items)\ndef test_share():\n    \"stars and bars\"\n    assert share(5, 3) == 21 and share(5, 3, 1) == 6 and share(3, 2, 2) == 0 and share(0, 4) == 1\n    assert share_capped(5, 3, 2) == 3 and share_capped(10, 2, 3) == 0\n    for items in range(0, 10):\n        for people in range(1, 5):\n            assert share(items, people) == brute(items, people, 0, items)\n            assert share(items, people, 1) == brute(items, people, 1, items)\n            for cap in range(0, 5):\n                assert share_capped(items, people, cap) == brute(items, people, 0, cap), (items, people, cap)")

### Exercise: None of the above

Write `count_none(N, divisors)`: how many of 1 to N are divisible by none of the divisors, by inclusion-exclusion over subsets (use `math.lcm`). Then `derangements(n)`: the orders of n items with nobody in their own place, with the recurrence, and `derangements_ie(n)`: the same by the inclusion-exclusion sum.

In [ ]:
from math import lcm, comb, factorial

def count_none(N, divisors):
    ___

def derangements(n):
    ___

def derangements_ie(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_include():\n    \"inclusion-exclusion\"\n    assert count_none(100, [2, 3, 5]) == 26 and count_none(10, []) == 10 and count_none(30, [4, 6]) == 20\n    rng = random.Random(2)\n    for _ in range(60):\n        divs = rng.sample(range(2, 15), rng.randint(0, 4))\n        N = rng.randint(1, 500)\n        assert count_none(N, divs) == sum(1 for x in range(1, N + 1) if all(x % d for d in divs))\n    assert [derangements(n) for n in range(8)] == [1, 0, 1, 2, 9, 44, 265, 1854]\n    for n in range(0, 8):\n        assert derangements(n) == sum(1 for p in itertools.permutations(range(n)) if all(p[i] != i for i in range(n)))\n    assert all(derangements(n) == derangements_ie(n) for n in range(60))")

### Exercise: Where in the list?

Write `rank(perm)`: the position (from 0) of a permutation of distinct items in sorted order, and `unrank(items, r)`: the permutation of `items` at position r, raising `ValueError` if r is out of range. Neither may list permutations. Then `next_perm(perm)`: the one after it (the last wraps to the first).

In [ ]:
from math import factorial

def rank(perm):
    ___

def unrank(items, r):
    ___

def next_perm(perm):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools\ndef test_rank():\n    \"factorial number system\"\n    assert rank(\"abc\") == 0 and rank(\"cba\") == 5 and rank([3, 1, 4, 2]) == 13\n    assert unrank(\"abc\", 3) == list(\"bca\") and unrank([1, 2, 3, 4], 13) == [3, 1, 4, 2]\n    for n in range(1, 7):\n        items = list(range(n))\n        for r, p in enumerate(itertools.permutations(items)):\n            assert rank(list(p)) == r and unrank(items, r) == list(p)\n    assert next_perm([1, 3, 2]) == [2, 1, 3] and next_perm([3, 2, 1]) == [1, 2, 3]\n    for bad in (-1, 6):\n        try:\n            unrank(\"abc\", bad)\n            assert False\n        except ValueError:\n            pass\n    big = list(range(20))\n    assert rank(unrank(big, 10 ** 17)) == 10 ** 17\n    assert \"permutations\" not in source")

### Bug hunt: Dividing too early

`c = c // (i + 1) * (n - i)` divides **before** multiplying, so it rounds down partway, and row 5 comes out as `[1, 5, 8, 6, 2, 0]`. Multiply first, `c * (n - i) // (i + 1)`, so it prints `[1, 5, 10, 10, 5, 1]`.

In [ ]:
n = 5
row = []
for k in range(n + 1):
    c = 1
    for i in range(k):
        c = c // (i + 1) * (n - i)
    row.append(c)
print(row)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"exact steps\"\n    expect_output(\"[1, 5, 10, 10, 5, 1]\")")

### Bug hunt: Every sign the same

Inclusion-exclusion alternates: subsets of an odd size are subtracted, of an even size added. This subtracts every non-empty subset, so it over-corrects and prints `-38` for the numbers up to 100 divisible by none of 2, 3 and 5. Fix the sign so it prints `26`.

In [ ]:
from math import lcm

divisors = [2, 3, 5]
total = 100
for mask in range(1, 1 << 3):
    chosen = [divisors[i] for i in range(3) if mask >> i & 1]
    total -= 100 // lcm(*chosen)
print(total)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"alternating\"\n    expect_output(\"26\")")

### Bug hunt: Bars in the wrong place

Sharing 5 sweets among 3 children uses 5 stars and **2** bars: C(5 + 3 − 1, 3 − 1). This uses k bars instead of k − 1 and prints `56` instead of `21`.

In [ ]:
from math import comb

sweets, children = 5, 3
print(comb(sweets + children, children))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"k minus one bars\"\n    expect_output(\"21\")")

### Puzzle: Inverse factorials

Arrange the lines to build factorials and inverse factorials mod p up to N. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the number of 4-digit PINs (digits 0 to 9, repeats allowed)
q = ___   # the ways to choose 3 toppings from 8, order ignored
r = ___   # the ways to share 4 identical sweets among 2 children
s = ___   # the fourth Catalan number, C3 (counting from C0 = 1)
t = ___   # the number of derangements of 4 items

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 10000 and q == 56 and r == 5, \"p, q and r\"\n    assert s == 5 and t == 9, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: routes around roadworks

A city grid has corners (0, 0) to (rows, cols); a courier moves only **east** (column + 1) or **south** (row + 1) from (0, 0) to (rows, cols). Some corners are closed for roadworks. Count the routes avoiding them, **mod 10⁹ + 7**, when the grid can be 100,000 × 100,000 (far too big for Week 53's grid table) but there are at most a few hundred closed corners.

Without obstacles, there are C(rows + cols, rows) routes. Sort the closed corners (by row, then column) and let `bad[i]` be the number of routes from the start to closed corner i that pass through **no earlier** closed corner: all routes to it minus, for every earlier closed corner j that can reach it, `bad[j]` × (routes from j to i). The answer is all routes minus the sum of `bad[i]` × (routes from i to the end). It's inclusion-exclusion organised as a DP: O(k²) for k closed corners, after O(rows + cols) for the factorial tables.

Write `routes(rows, cols, closed)`, and `routes_small(rows, cols, closed)`, the grid DP from Week 53 (mod 10⁹ + 7), for checking.

In [ ]:
def routes(rows, cols, closed):
    ___

def routes_small(rows, cols, closed):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef test_routes():\n    \"obstacles by counting\"\n    assert routes(2, 2, []) == 6 and routes(2, 2, [(1, 1)]) == 2 and routes(1, 1, [(0, 1), (1, 0)]) == 0\n    assert routes(3, 3, [(0, 0)]) == 0 and routes(3, 3, [(3, 3)]) == 0\n    rng = random.Random(3)\n    for _ in range(80):\n        r, c = rng.randint(0, 8), rng.randint(0, 8)\n        closed = list({(rng.randint(0, r), rng.randint(0, c)) for _ in range(rng.randint(0, 6))})\n        assert routes(r, c, closed) == routes_small(r, c, closed), (r, c, closed)\n    big = [(rng.randint(1, 99_999), rng.randint(1, 99_999)) for _ in range(300)]\n    t0 = time.perf_counter()\n    answer = routes(100_000, 100_000, big)\n    assert 0 <= answer < 10 ** 9 + 7 and time.perf_counter() - t0 < 10\n    from math import comb\n    assert routes(2_000, 3_000, []) == comb(5_000, 2_000) % (10 ** 9 + 7)")

## Warm-up

Arranging 5 things: 5! = 120. Choosing an ordered pair from 5: 5 × 4 = 20. Choosing an unordered pair: 20 / 2 = 10. Row 4 of Pascal's triangle sums to 2⁴, and 2,598,960 five-card poker hands come out instantly. Python's integers are exact at any size: 1000! has 2,568 digits.

## Sum, product, order

Two rules build almost everything:

- **Rule of sum**: if a choice is one of several **separate** kinds, add their counts.
- **Rule of product**: if a choice is a **sequence** of independent steps, multiply the counts at each step.

From the product rule: there are n! orders of n things; P(n, k) = n × (n − 1) × ... × (n − k + 1) = n! / (n − k)! ordered selections of k; and since each **unordered** selection appears in k! orders, C(n, k) = n! / (k! (n − k)!). With repetition allowed, ordered: nᵏ (a 4-digit PIN: 10⁴).

The key question for any count: **does order matter, and can things repeat?**

| | No repeats | Repeats allowed |
|---|---|---|
| Order matters | P(n, k) | nᵏ |
| Order doesn't | C(n, k) | C(n + k − 1, k) (stars and bars, below) |

`math.comb`, `math.perm` and `math.factorial` compute these exactly.

## Binomials, big and modular

C(n, k) obeys **Pascal's rule**: C(n, k) = C(n − 1, k − 1) + C(n − 1, k) (the first item is in the selection or not). That's a DP that builds whole rows with only additions, which is perfect for counting mod anything. To compute one C(n, k) exactly, multiply and divide in turns: `c = c * (n - i) // (i + 1)` for i from 0 to k − 1 stays a whole number at every step (each intermediate value is itself a binomial).

**Mod a prime p** (with n < p), division isn't allowed directly, but yesterday's inverses fix that: precompute `fact[i]` = i! mod p and `inv_fact[i]` = the inverse of i! mod p, then C(n, k) = fact[n] × inv_fact[k] × inv_fact[n − k] mod p, in O(1) per query. Compute `inv_fact` with one `pow(fact[N], -1, p)` and then go downwards: inv_fact[i − 1] = inv_fact[i] × i.

## Stars and bars

How many ways can n identical sweets be shared among k children? Lay the n sweets (stars) in a row and place k − 1 dividers (bars) among them: each arrangement of n stars and k − 1 bars is one sharing. So the count is **C(n + k − 1, k − 1)**. If every child must get at least one, give them one each first: C(n − 1, k − 1).

The same count is the number of **multisets** of size n from k kinds (choosing with repetition, order ignored), and the number of solutions to x₁ + ... + x_k = n in non-negative whole numbers. (William Feller's 1950 probability textbook made the name popular.)

## Inclusion-exclusion

To count what has **none** of several properties, start with everything, subtract those with each property, add back those with each **pair** (subtracted twice), subtract each triple, and so on:

|none| = total − Σ|Aᵢ| + Σ|Aᵢ ∩ Aⱼ| − Σ|Aᵢ ∩ Aⱼ ∩ A_k| + ...

The sign is + for an even number of properties and − for an odd number. With m properties that's 2ᵐ terms, one per subset: loop over bitmasks (Week 54), with the sign from the bit count.

- Numbers up to N divisible by **none** of a list: for each subset, subtract or add N // lcm(subset).
- **Derangements** (orders where nobody is in their own place, like a secret Santa where no one draws themselves): D(n) = Σ (−1)ᵏ C(n, k) (n − k)!, about n!/e. Also D(n) = (n − 1)(D(n − 1) + D(n − 2)).

## Catalan numbers and ranking

The **Catalan numbers** 1, 1, 2, 5, 14, 42, 132, ... count the balanced bracket strings with n pairs, the shapes of binary trees with n nodes, the ways to triangulate a polygon, and many more. Cₙ = C(2n, n) / (n + 1). They're named after Eugène Catalan (1838), though Ming Antu used them in China in the 1730s.

**Ranking** a permutation means finding its position in sorted order without listing those before it. For each position from the left, count the unused items smaller than the one there: each such smaller item would start a whole block of (remaining)! permutations that come earlier. Add them up. **Unranking** reverses it: divide the position by the factorials, picking the item at each step. This is the **factorial number system** (Lehmer codes, after Derrick Lehmer).

## Practice

## Quiz

## Daily challenge

## Before you go

How many different five-card hands contain at least one ace? Count it two ways: directly with inclusion-exclusion, and as "all hands minus hands with no ace". Tomorrow: randomised algorithms, where a little luck buys a lot of speed.